# Two-Bot Conversation

In this notebook, we set up two chatbots with opposing personalities and let them talk to each other. One bot is argumentative and snarky, while the other is polite and tries to find common ground. We use a local LLM served through Ollama and stream the responses in real time.

## Imports

In [ ]:
import os
import requests
from openai import OpenAI
from IPython.display import Markdown, display, update_display

## Configuration

We point the OpenAI client at a local Ollama server. The `llama3.2:1b` model is a small, fast model that works well for this kind of conversational demo.

In [ ]:
API_KEY = "ollama"
BASE_URL = "http://localhost:11434/v1"
MODEL = "llama3.2:1b"

## Initialize the Client

In [ ]:
from openai import OpenAI
client = OpenAI(api_key=API_KEY,base_url=BASE_URL)

## Define the Bot Personalities

Each bot gets a system prompt that defines its behavior:

- **Bot 1** is argumentative and snarky. It disagrees with everything.
- **Bot 2** is polite and tries to agree or calm things down.

We also set up the initial messages to kick off the conversation.

In [ ]:
bot_1_sys = """You are a chatbot who is very argumentative;
you disagree with anything in the conversation and you challenge everything, in a snarky way.
Your answers shouldn't be long"""

bot_2_sys = """You are a very polite, courteous chatbot. You try to agree with
everything the other person says, or find common ground. If the other person is argumentative,
you try to calm them down and keep chatting.
Your answers should'nt be long"""

bot_1_messages = ["Hi there"]
bot_2_messages = ["Hi"]

## Bot Response Functions

Each function builds the message history from the perspective of its own bot and calls the model with streaming enabled. Bot 1 treats its own messages as the assistant and Bot 2's messages as the user, and vice versa.

In [ ]:
def call_bot_1():
    messages = [{"role": "system", "content": bot_1_sys}]
    for bot1 , bot2 in zip(bot_1_messages,bot_2_messages):
        messages.append({"role": "assistant", "content": bot1})
        messages.append({"role": "user", "content": bot2})
    response = client.chat.completions.create(model=MODEL,messages=messages,stream=True,max_tokens=100)
    return response

In [ ]:
def call_bot_2():
    messages = [{"role": "system", "content": bot_2_sys}]
    for bot1 , bot2 in zip(bot_1_messages,bot_2_messages):
        messages.append({"role": "user", "content": bot1})
        messages.append({"role": "assistant", "content": bot2})
    if len(bot_1_messages) > len(bot_2_messages):
        messages.append({"role": "user", "content": bot_1_messages[-1]})
    response = client.chat.completions.create(model=MODEL,messages=messages,stream=True,max_tokens=100)
    return response

## Run the Conversation

The loop below runs 5 rounds of back-and-forth between the two bots. Each response is streamed token by token and displayed live using `update_display`, so you can watch the conversation unfold in real time.

In [ ]:
bot_1_messages = ["Hi there"]
bot_2_messages = ["Hi"]

display(Markdown(f"### Bot 1:\n{bot_1_messages[0]}\n"))
display(Markdown(f"### Bot 2:\n{bot_2_messages[0]}\n"))

response = ""
display_handle = display(Markdown(""), display_id=True)

for i in range(5):
    bot1_text = ""
    response += f"### Bot 1:\n"
    bot1_next = call_bot_1()
    for chunk in bot1_next:
        content = chunk.choices[0].delta.content or ''
        bot1_text += content
        response += content
        update_display(Markdown(response), display_id=display_handle.display_id)
    response += "\n\n"
    bot_1_messages.append(bot1_text)

    bot2_text = ""
    response += f"### Bot 2:\n"
    bot2_next = call_bot_2()
    for chunk in bot2_next:
        content = chunk.choices[0].delta.content or ''
        bot2_text += content
        response += content
        update_display(Markdown(response), display_id=display_handle.display_id)
    response += "\n\n"
    bot_2_messages.append(bot2_text)